# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all key entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print top-level metadata fields
metadata = dataset.metadata
print("Dataset Name:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))
print("Identifier:", getattr(metadata, 'identifier', None))
print("Version:", getattr(metadata, 'version', None))

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All identifiers are referenced by their `@id`, as per Croissant specifications.

In [ ]:
# List all available RecordSets by their @id and name
print("Available RecordSets:")
record_sets = []
for rs in getattr(metadata, 'record_sets', []):
    print(f"@id: {getattr(rs, '@id', None)}  |  name: {getattr(rs, 'name', None)}")
    record_sets.append(getattr(rs, '@id', None))
if not record_sets:
    # Try the property 'recordSet' as sometimes it's singular in Croissant schemas
    for rs in getattr(metadata, 'record_set', []):
        print(f"@id: {getattr(rs, '@id', None)}  |  name: {getattr(rs, 'name', None)}")
        record_sets.append(getattr(rs, '@id', None))
print()

for rs in getattr(metadata, 'record_sets', []) + getattr(metadata, 'record_set', []):
    rs_id = getattr(rs, '@id', None)
    print(f"Fields for RecordSet {rs_id}:")
    if hasattr(rs, 'fields'):
        for field in getattr(rs, 'fields', []):
            print(f"  - Field @id: {getattr(field, '@id', None)} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview. Below, we extract all record sets if available.

In [ ]:
# Gather RecordSet @ids
record_sets = []
for rs in getattr(metadata, 'record_sets', []):
    record_sets.append(getattr(rs, '@id', None))
for rs in getattr(metadata, 'record_set', []):
    record_sets.append(getattr(rs, '@id', None))
record_sets = [r for r in record_sets if r]

dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from RecordSet @id='{record_set_id}'. Columns: {list(df.columns)}")
        else:
            print(f"No records found for RecordSet @id='{record_set_id}'.")
    except Exception as e:
        print(f"Could not load RecordSet @id='{record_set_id}': {e}")
print()
# Show a sample of one DataFrame
if dataframes:
    sample_rs = list(dataframes.keys())[0]
    print(f"Sample data from RecordSet @id='{sample_rs}':")
    display(dataframes[sample_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalization, and basic aggregations using field `@id`s.

In [ ]:
# For this example, we'll operate on the first available RecordSet and a likely numeric field (e.g. molecular weight, or peptide counts).
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Columns in RecordSet @id='{record_set_id}':", df.columns.tolist())

    # Try to find a sensible numeric field, e.g. 'molecular_weight', 'MW', 'peptide_counts', etc.
    numeric_field = None
    candidate_fields = [c for c in df.columns if any(x in c.lower() for x in ['weight', 'mw', 'count', 'abundance', 'coverage', 'number'])]
    for c in candidate_fields:
        # Test if the column is numeric
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field = c
            break

    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field: look for a likely grouping field
        group_field = None
        for c in df.columns:
            if c != numeric_field and df[c].nunique() > 1 and df[c].nunique() < (0.5 * len(df)):
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals() and numeric_field:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If we found a group_field, show boxplot
    if 'group_field' in locals() and group_field:
        fig, ax = plt.subplots(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df, ax=ax)
        ax.set_title(f"{numeric_field} grouped by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we have loaded and explored the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant`.

- All data entities (record sets, fields) were referenced via their Croissant `@id`s, ensuring full transparency and traceability.
- We successfully loaded metadata, listed available record sets, extracted records, performed basic filtering and normalization on a numeric attribute, and visualized distributions.

Further domain-specific biostatistics or machine learning analysis can now be conducted using the structured DataFrame(s) prepared from this FAIR²-compliant dataset.